# 02 — Saving resources with time instead of space

**Narrative point 2.** Idle qubits are usually protected by more space: a larger distance,
or a denser storage code. The optimal interval protects them with time: the same code,
extracted less often. Two ways to price that, from the H2-1E regime grid (notebook 08):

- **qubits:** does the optimal interval at distance d beat back-to-back extraction at d + 2?
  If so, waiting replaces a distance step: (d+2)² − d² data qubits per logical qubit
  (rotated surface code; about twice that with ancillas);
- **operations:** per unit time the memory extracts Δt* times less often, so it runs Δt*
  times fewer two-qubit gates and measurements, which dominate a trapped-ion machine's cost
  (Quantinuum's HQC formula weights them; *interp.*).

In [ ]:
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from qec_timing import plotting as P
P.style()
R = Path("..") / "results"

In [ ]:
grid = json.loads(next((R / "regime" / "grid").glob("regime_*.json")).read_text())
rows = grid["rows"]
T_R = sorted({r["t_round"] for r in rows}); DS = sorted({r["d"] for r in rows})
cell = {(r["t_round"], r["d"]): r for r in rows}
def best(r):                            # lowest used rate and its interval
    return min((p["r"], p["dt"]) for p in r["points"] if p.get("used"))
print("calibration:", grid["calibration"]["source"][:90], "...")
print(f"{len(rows)} (t_R, d) cells; rounds per run {grid['setup']['rounds']}, "
      f"target {grid['setup']['target']} failures per point, max {grid['setup']['max_shots']:,} shots")

In [ ]:
print(f"{'t_R ms':>7}{'d':>3}{'R b2b(d)':>11}{'R opt(d)':>11}{'R b2b(d+2)':>12}   {'optimal(d) beats b2b(d+2)?':<30}{'extractions':>12}")
for t in T_R:
    for d in DS[:-1]:
        a, b = cell.get((t, d)), cell.get((t, d + 2))
        if not (a and b):
            continue
        r_b2b, r_opt, r_next = a["points"][0]["r"], best(a)[0], b["points"][0]["r"]
        verdict = f"yes: saves {(d + 2)**2 - d**2} data qubits" if r_opt < r_next else "no"
        print(f"{t*1e3:>7.1f}{d:>3}{r_b2b:>11.2e}{r_opt:>11.2e}{r_next:>12.2e}   {verdict:<30}"
              f"{a['dt_star_rounds']:>9.1f}x fewer")

In [ ]:
from matplotlib.lines import Line2D
fig, ax = plt.subplots(figsize=(7.5, 3.8))
for d in DS:
    ts = [t for t in T_R if (t, d) in cell]
    ax.plot([t * 1e3 for t in ts], [cell[(t, d)]["points"][0]["r"] for t in ts], "--", color=P.D_COLOR[d], lw=1.5)
    ax.plot([t * 1e3 for t in ts], [best(cell[(t, d)])[0] for t in ts], "o-", color=P.D_COLOR[d])
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("$t_R$ (ms)"); ax.set_ylabel("logical error rate per unit time")
h = [Line2D([], [], color=P.D_COLOR[d], marker="o", label=f"d = {d}") for d in DS]
h += [Line2D([], [], color=P.MUTED, ls="--", label="back to back"),
      Line2D([], [], color=P.MUTED, marker="o", label="optimal interval")]
ax.legend(handles=h, fontsize=8, ncol=2)
ax.set_title("Optimal interval at d against back-to-back extraction at d + 2", loc="left")
fig.tight_layout()
P.save(fig, "02_time_for_space_1")

**Reading.**
- *Measured:* where the table says "yes", the optimal interval at distance d gives a lower
  logical error rate per unit time than back-to-back extraction at d + 2. Time replaces a
  distance step, saving (d+2)² − d² data qubits per idle logical qubit (16 from d = 3 to 5,
  24 from 5 to 7), with no change to the code.
- *Measured:* in the same regime the memory extracts Δt* times less often, tens of times at
  t_R of a few ms, so its gate and measurement count per unit time falls by that factor.
- *Interp., the pitch:* combined with notebook 01 (most logical qubit time in sequential
  workloads is idle, in long stretches), the interval is a free knob acting on most of an
  application's space-time volume. Mis-setting it costs more as d grows (the paper's Γ_opt
  grows exponentially in α ∝ d), so it should be calibrated per device, which takes one
  calibration (notebook 06).
- *Boundary:* the saving exists only where idle error per round is well below p2. On H2 that
  depends on t_R, unknown until measured (notebooks 08, 09).